# Spark Avatar Studio

Connect to the DGX Spark, download the lip-sync and background models, capture photos or an idle video of yourself, build the avatar, turn text into speech with your cloned voice, render the talking avatar, replace the background, and tune the result. Each section runs one of the `stepN_*.py` scripts; edit the `ARGS` line and run the cell. Steps 4 and 9 prompt through `input()`, which Jupyter provides. Step 4 needs a camera on the machine running the notebook.

Edit `studio.yaml` first: Spark host, user, key, your name as owner, and the background you want.

In [ ]:
import subprocess, sys, json
from pathlib import Path
ROOT = Path.cwd()

def run(script, args=''):
    """Run a step script with live output in the notebook."""
    cmd = [sys.executable, str(ROOT / script), *args.split()]
    print('$', ' '.join(cmd[1:]))
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=ROOT)
    for line in p.stdout: print(line, end='')
    return p.wait()

def status():
    p = ROOT / 'state.json'
    print(json.dumps(json.loads(p.read_text()), indent=1) if p.exists() else 'no state yet')

status()

## Step 1 - Connect to the DGX Spark and prepare the avatar environment.

Checks SSH and the GPU, creates the studio venv on the Spark with the cu130 PyTorch wheels, clones LatentSync
(Apache 2.0) with its torch pins stripped so it cannot replace the Spark's PyTorch, installs the packages
(onnxruntime CPU, face-alignment, transformers for the matting model), syncs this folder, and records the
result in state.json.

    python step1_connect_spark.py            # connect, check, install what is missing
    python step1_connect_spark.py --check    # connect and report only

In [ ]:
ARGS = ''
run('step1_connect_spark.py', ARGS)

## Step 2 - Download the models the avatar needs, on the Spark.

* LatentSync 1.6 lip-sync checkpoint (latentsync_unet.pt, about 5 GB) and its Whisper-tiny audio encoder
  * stabilityai/sd-vae-ft-mse, the VAE LatentSync loads on first run (fetched now so a render never stalls)
  * BiRefNet, the background-removal model (about 0.9 GB)
Everything lands under models/ on the Spark and is linked into the LatentSync checkpoints folder.
Wav2Lip (fast, lower quality) is optional: its weights sit behind a request form, so --wav2lip only prints where to put them.

    python step2_download_models.py
    python step2_download_models.py --only matting
    python step2_download_models.py --wav2lip

In [ ]:
ARGS = ''
run('step2_download_models.py', ARGS)

## Step 3 - Load the models into the Spark's memory and prove they fit.

Loads the BiRefNet matting model onto the GPU and times one 1024-pixel frame; confirms the LatentSync checkpoint
files are present and the right size; reports unified-memory use. LatentSync itself loads per render (its inference
script builds the pipeline each run, about 30 to 60 s), so the proof for it is --smoke: a 3 s tone is lip-synced onto
your avatar with 10 steps. Run --smoke after step 5.

    python step3_load_models.py
    python step3_load_models.py --smoke

In [ ]:
ARGS = ''
run('step3_load_models.py', ARGS)

## Step 4 - Capture photos and an idle video of yourself, check them, upload them.

Runs where the camera is (your Mac). The avatar is built from what you give it here:
  * best: a 30 to 60 s idle video (--video) - you looking at the lens, relaxed, blinking, small natural movement,
    mouth mostly closed. LatentSync keeps your head movement and rewrites only the mouth, so real footage looks alive.
  * enough to start: photos only (--photos). One good front-facing photo works; the avatar will have a still head with
    gentle camera drift. Five shots are requested so you can pick the best and keep spares.
Every file is checked (one face, face large enough, sharp, evenly lit, centred) and reported before upload.

    python step4_capture_media.py --i-am-the-subject --photos          # guided stills from the webcam
    python step4_capture_media.py --i-am-the-subject --video           # idle take, studio.yaml capture.video_seconds
    python step4_capture_media.py --i-am-the-subject --import me.jpg idle.mov
    python step4_capture_media.py --check-only                         # re-run the checks on what is already saved

In [ ]:
ARGS = '--i-am-the-subject --video'
run('step4_capture_media.py', ARGS)

## Step 5 - Build the avatar from your photos or idle video.

Runs on the Spark (the Mac side just starts it). From avatars/<name>/source/ it makes:
  idle_src.mp4   your footage normalised to constant 25 fps, even dimensions, at most 1080 lines
  avatar.json    consent record, mode (video or photo), loop window, size, fps
  bg_preview.png the first second composited on the background you configured, so you can judge the matte before any render
With a video, it finds the loop window: the pair of moments, at least 6 s apart, where your face looks most alike, so the
ping-pong footage under long speech has no visible jump. With photos only, it makes 12 s of still head with a gentle drift.

    python step5_build_avatar.py --i-am-the-subject
    python step5_build_avatar.py --i-am-the-subject --background gradient     # preview another background
    python step5_build_avatar.py --i-am-the-subject --photo photo_front_neutral.png

In [ ]:
ARGS = '--i-am-the-subject'
run('step5_build_avatar.py', ARGS)

## Step 6 - Turn text into the speech the avatar will lip-sync.

Sends each paragraph of a text file to the Spark Voice Studio model server (your cloned or fine-tuned voice), joins the
clips with short pauses, and writes data/audio/<name>.wav plus <name>.srt (one caption per paragraph) and <name>.json.
Or bring your own recording: --audio file.wav copies it in unchanged.

    python step6_make_audio.py lesson.txt
    python step6_make_audio.py lesson.txt --pause 0.6 --language English
    python step6_make_audio.py --audio my_recording.wav --name intro
Start the voice server first (Spark Voice Studio step 3).

In [ ]:
ARGS = 'sample.txt'
run('step6_make_audio.py', ARGS)

## Step 7 - Render the talking avatar: audio in, lip-synced video out, optional new background.

Mac side: uploads the audio, runs the render on the Spark, downloads the results to out/<name>/.
Spark side (--here): cuts the audio at silences into chunks of at most render.max_chunk_seconds, builds ping-pong footage
for each chunk from your avatar's loop window (the footage cursor carries on from chunk to chunk), runs LatentSync 1.6 on
each, joins the chunks, lays your original audio back on (full quality, unshifted), then runs the background stage.
Chunks are cached by hash: re-run after an interruption or a setting change and only the affected chunks redo.

Outputs in out/<name>/:   <name>_raw.mp4 (lip-synced, original background) and <name>_final.mp4 (after the background stage).
Because the raw file is kept, step 8 can change the background in minutes without repeating the lip-sync.

    python step7_render_avatar.py --i-am-the-subject --audio data/audio/lesson.wav
    python step7_render_avatar.py --i-am-the-subject --audio lesson.wav --background gradient --steps 30
    python step7_render_avatar.py --i-am-the-subject --tone 3 --steps 10 --name smoke     # smoke test, no audio needed

In [ ]:
ARGS = '--i-am-the-subject --audio data/audio/sample.wav'
run('step7_render_avatar.py', ARGS)

## Step 8 - Remove or replace the background of a rendered avatar video (no new lip-sync).

Takes out/<name>/<name>_raw.mp4 (or any video) and writes out/<name>/<name>_<mode>.mp4. Choose:
  keep         leave the room as filmed
  transparent  alpha channel: ProRes 4444 .mov (editors) or VP9 .webm (browsers); drop it over slides in your editor
  color        one flat colour, e.g. brand navy
  gradient     two-colour vertical blend
  image        a still: slide, office, brand backdrop (cover-fitted)
  video        a looping clip behind you
  blur         the original room, blurred and darkened
Engines: birefnet (AI matte, works on any room) or chroma (green/blue screen key: cleaner edges if you filmed on a screen).
Edge tuning: --choke pulls the edge inward against halos of the old room; --alpha-smooth steadies flicker; --feather softens.

    python step8_background.py smoke --mode color --color "#0B1F3A"
    python step8_background.py lesson --mode image --image backdrops/office.jpg
    python step8_background.py lesson --mode transparent --webm
    python step8_background.py lesson --mode video --video loops/particles.mp4 --engine chroma --chroma-key "#00B140"

In [ ]:
ARGS = 'lesson --mode color --color #0B1F3A'
run('step8_background.py', ARGS)

## Step 9 - Assess the avatar and tune it.

A lip-sync avatar has no per-person training: LatentSync follows your footage and your audio. What you can tune are the
render settings and the footage, and this step finds the best settings by experiment on a short excerpt.

    python step9_assess_tune.py --auto lesson                 # automatic checks: length match, black frames, flicker spikes
    python step9_assess_tune.py --play lesson                 # watch it, score sync / likeness / artifacts / background 1-5
    python step9_assess_tune.py --sweep offset --audio data/audio/lesson.wav     # 5 renders of a 6 s excerpt, lips earlier/later
    python step9_assess_tune.py --sweep quality --audio data/audio/lesson.wav    # steps 15/25 x guidance 1.5/2.0
    python step9_assess_tune.py --apply tune_offset_p04       # make that variant's settings the default for step 7
    python step9_assess_tune.py --report
If lips are consistently early or late, sweep offset. If the mouth looks mushy, sweep quality. If the head looks frozen or
the loop jumps, the fix is the footage (step 4: record 30 to 60 s, not a photo).

In [ ]:
ARGS = '--report'
run('step9_assess_tune.py', ARGS)

## Look at the result

Play a finished render, or the background preview from step 5, without leaving the notebook.

In [ ]:
from IPython.display import Video, Image, display
name = 'lesson'   # the render name
p = ROOT / 'out' / name / f'{name}_final.mp4'
print(p, 'exists' if p.exists() else 'not rendered yet')
display(Video(str(p), embed=False, width=640)) if p.exists() else None
from studio.common import cfg
bp = ROOT / 'avatars' / cfg()['avatar']['name'] / 'bg_preview.png'
display(Image(str(bp))) if bp.exists() else print('no background preview yet (step 5)')

## Assessment report

Scores per render with the settings that produced them, so a sweep from step 9 shows which settings won.

In [ ]:
run('step9_assess_tune.py', '--report')